# ELEC5308 — Lab 8: Trajectory Control & Localisation

---

## Structure

| Section | Topic | Time | Points |
|---------|-------|------|--------|
| **Mandatory** | Pure Pursuit lateral controller | ~20 min | 20 pts |
| **Optional A** | Odometry drift accumulation | open | 5 pts |
| **Optional B** | Sensor-fusion predict–correct | open | 5 pts |
| **Optional C** | 2-D ICP scan matching | open | 5 pts |
| **Optional D** | Point-cloud de-skewing | open | 5 pts |

> **Submission:** Upload this `.ipynb` to Gradescope.  
> Do **not** modify any cell marked `# DO NOT EDIT`.  
> Replace every `# YOUR CODE HERE` block and remove the `raise NotImplementedError()` line.



In [19]:
# DO NOT EDIT — imports and shared fixtures
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

WAYPOINTS = np.array([
    [  0.0,   0.0],
    [  5.0,   0.3],
    [ 10.0,   0.5],
    [ 15.0,   0.6],
    [ 20.0,   0.5],
    [ 25.0,   0.2],
    [ 30.0,  -0.3],
    [ 35.0,  -0.9],
    [ 40.0,  -1.2],
    [ 45.0,  -1.0],
    [ 50.0,  -0.4],
    [ 55.0,   0.4],
    [ 60.0,   1.2],
    [ 65.0,   1.8],
    [ 70.0,   2.0],
])

WHEELBASE   = 2.5     # metres
DELTA_MAX   = np.deg2rad(30)   # maximum steering angle
DT          = 0.1     # simulation time step (s)
SPEED       = 5.0     # constant forward speed (m/s)

print("Fixtures loaded. Waypoints:", WAYPOINTS.shape)


Fixtures loaded. Waypoints: (15, 2)


---
# ★ MANDATORY — Pure Pursuit Lateral Controller  *(20 pts total)*

Pure Pursuit steers toward a **lookahead point** P on the path at distance $L_d$ ahead of the front axle, rather than correcting the current lateral error directly.

$$\delta = \arctan\!\left(\frac{2L\sin\alpha}{L_d}\right)$$

where $\alpha$ is the heading angle to P and $L$ is the wheelbase.

You will implement the controller in three self-contained functions, then verify it on the campus path.


## Task 1 — Select the lookahead point  *(5 pts)*

Scan `waypoints` in order and return the **index** of the **first** waypoint whose Euclidean distance from `pos` is ≥ `Ld`.  
If no such waypoint exists, return the index of the **last** waypoint.

**Signature:** `select_lookahead_point(waypoints, pos, Ld) → int`


In [20]:
def select_lookahead_point(waypoints: np.ndarray,
                           pos: np.ndarray,
                           Ld: float) -> int:
    """
    Return the index of the first waypoint at distance >= Ld from pos.

    Parameters
    ----------
    waypoints : np.ndarray, shape (N, 2)
        Path waypoints [[x0, y0], [x1, y1], ...].
    pos : np.ndarray, shape (2,)
        Current front-axle position [x, y] in metres.
    Ld : float
        Lookahead distance in metres (> 0).

    Returns
    -------
    int
        Index of the chosen lookahead waypoint (0 <= idx < N).
    """
    waypoints = np.asarray(waypoints, dtype=float)
    pos = np.asarray(pos, dtype=float)
    dists = np.linalg.norm(waypoints - pos, axis=1)
    idx = np.flatnonzero(dists >= Ld)
    return int(idx[0]) if idx.size > 0 else len(waypoints) - 1


In [21]:
# ── Autograder tests — Task 1 (5 pts) ──────────────────────────────────── #
# DO NOT EDIT

from numpy.testing import assert_equal

# Test 1a: standard case — third waypoint is the first one at dist >= 8 m
_wp = np.array([[0.,0.],[3.,0.],[8.,0.],[15.,0.]])
_pos = np.array([0., 0.])
assert_equal(select_lookahead_point(_wp, _pos, 8.0), 2,
             err_msg="Should return index 2 ([8,0] is the first wp at dist >= 8)")

# Test 1b: all waypoints inside Ld → return last
_wp2 = np.array([[1.,0.],[2.,0.],[3.,0.]])
_pos2 = np.array([0., 0.])
assert_equal(select_lookahead_point(_wp2, _pos2, 100.0), 2,
             err_msg="All waypoints inside Ld; should return last index")

# Test 1c: first waypoint already satisfies Ld
_wp3 = np.array([[10.,0.],[20.,0.]])
_pos3 = np.array([0., 0.])
assert_equal(select_lookahead_point(_wp3, _pos3, 5.0), 0,
             err_msg="First waypoint at dist=10 satisfies Ld=5; return index 0")

# Test 1d: uses the campus fixture
idx = select_lookahead_point(WAYPOINTS, np.array([0., 0.]), Ld=10.0)
assert 0 <= idx < len(WAYPOINTS), "Index out of range"
assert np.linalg.norm(WAYPOINTS[idx] - np.array([0., 0.])) >= 10.0,     "Selected waypoint must be at distance >= Ld"

print("Task 1 — all assertions passed ✓")


Task 1 — all assertions passed ✓


## Task 2 — Compute the heading angle α  *(5 pts)*

$\alpha$ is the signed angle from the vehicle's current heading to the direction of the lookahead point, wrapped to $[-\pi,\,\pi]$.

$$\alpha = \text{atan2}(\Delta y, \Delta x) - \psi$$

where $\psi$ is the vehicle heading and $(\Delta x, \Delta y) = P - x_{\text{pos}}$.

**Signature:** `compute_alpha(lookahead_wp, pos, heading) → float`


In [22]:
def compute_alpha(lookahead_wp: np.ndarray,
                  pos: np.ndarray,
                  heading: float) -> float:
    """
    Compute the heading angle alpha from the vehicle to the lookahead point.

    Parameters
    ----------
    lookahead_wp : np.ndarray, shape (2,)
        Lookahead point [x, y] in world frame.
    pos : np.ndarray, shape (2,)
        Vehicle front-axle position [x, y].
    heading : float
        Vehicle heading psi in radians (measured CCW from +x axis).

    Returns
    -------
    float
        alpha in radians, wrapped to [-pi, pi].
    """
    dx, dy = np.asarray(lookahead_wp, dtype=float) - np.asarray(pos, dtype=float)
    alpha = np.arctan2(dy, dx) - heading
    return float(np.arctan2(np.sin(alpha), np.cos(alpha)))


In [23]:
# ── Autograder tests — Task 2 (5 pts) ──────────────────────────────────── #
# DO NOT EDIT

from numpy.testing import assert_allclose

# Test 2a: vehicle heading east (+x), lookahead at 45° NE → alpha = pi/4
alpha = compute_alpha(np.array([1., 1.]), np.array([0., 0.]), heading=0.0)
assert_allclose(alpha, np.pi / 4, atol=1e-9,
                err_msg="Heading east, wp at 45°NE: alpha should be pi/4")

# Test 2b: vehicle heading north (+y), lookahead directly east → alpha = -pi/2
alpha = compute_alpha(np.array([1., 0.]), np.array([0., 0.]), heading=np.pi/2)
assert_allclose(alpha, -np.pi/2, atol=1e-9,
                err_msg="Heading north, wp directly east: alpha should be -pi/2")

# Test 2c: lookahead directly ahead (alpha = 0)
alpha = compute_alpha(np.array([5., 0.]), np.array([0., 0.]), heading=0.0)
assert_allclose(alpha, 0.0, atol=1e-9,
                err_msg="Lookahead dead ahead should give alpha=0")

# Test 2d: wrap-around — heading nearly west, wp slightly north of east
alpha = compute_alpha(np.array([1., 0.]), np.array([0., 0.]), heading=np.pi - 0.01)
assert -np.pi <= alpha <= np.pi, "alpha must be in [-pi, pi]"

print("Task 2 — all assertions passed ✓")


Task 2 — all assertions passed ✓


## Task 3 — Compute the steering command δ  *(5 pts)*

Use the Pure Pursuit formula and **clip** to the actuator limit:

$$\delta = \text{clip}\!\left(\arctan\!\left(\frac{2L\sin\alpha}{L_d}\right),\; -\delta_{\max},\; +\delta_{\max}\right)$$

**Signature:** `compute_steering(alpha, Ld, wheelbase, delta_max) → float`


In [24]:
def compute_steering(alpha: float,
                     Ld: float,
                     wheelbase: float,
                     delta_max: float) -> float:
    """
    Compute the Pure Pursuit steering angle, clipped to actuator limits.

    Parameters
    ----------
    alpha     : float  Heading angle to lookahead point (radians).
    Ld        : float  Lookahead distance (metres, > 0).
    wheelbase : float  Vehicle wheelbase L (metres).
    delta_max : float  Maximum steering angle (radians, > 0).

    Returns
    -------
    float
        Steering angle delta in radians, within [-delta_max, +delta_max].
    """
    delta = np.arctan(2.0 * wheelbase * np.sin(alpha) / Ld)
    return float(np.clip(delta, -delta_max, delta_max))


In [25]:
# ── Autograder tests — Task 3 (5 pts) ──────────────────────────────────── #
# DO NOT EDIT

from numpy.testing import assert_allclose

# Test 3a: alpha=0 (dead ahead) → delta=0 regardless of parameters
delta = compute_steering(0.0, 5.0, 2.5, np.pi/4)
assert_allclose(delta, 0.0, atol=1e-12,
                err_msg="alpha=0 must give delta=0")

# Test 3b: known value — alpha=pi/6, Ld=5, L=2.5
alpha = np.pi / 6
expected = np.arctan(2 * 2.5 * np.sin(alpha) / 5.0)
delta = compute_steering(alpha, 5.0, 2.5, np.pi)
assert_allclose(delta, expected, atol=1e-9,
                err_msg="Incorrect steering formula")

# Test 3c: saturation — very large alpha must saturate at delta_max
delta_max = np.deg2rad(25)
delta = compute_steering(np.pi/2, 1.0, 2.5, delta_max)
assert_allclose(delta, delta_max, atol=1e-9,
                err_msg="Should saturate at +delta_max")

# Test 3d: symmetric saturation — negative alpha
delta = compute_steering(-np.pi/2, 1.0, 2.5, delta_max)
assert_allclose(delta, -delta_max, atol=1e-9,
                err_msg="Should saturate at -delta_max")

# Test 3e: output is within bounds for arbitrary alpha
for a in np.linspace(-np.pi, np.pi, 50):
    d = compute_steering(a, 5.0, 2.5, DELTA_MAX)
    assert -DELTA_MAX - 1e-12 <= d <= DELTA_MAX + 1e-12,         f"delta={d:.4f} exceeds delta_max for alpha={a:.3f}"

print("Task 3 — all assertions passed ✓")


Task 3 — all assertions passed ✓


## Integration check — Close-loop simulation on the campus path  *(5 pts)*

The cell below runs a kinematic bicycle simulation.  
**Your three functions must be implemented correctly** for this to pass.  
The test checks that the vehicle stays within **2.0 m** of the path at all times.  
You do **not** need to write any code here — just run the cell.


In [26]:
# DO NOT EDIT — kinematic bicycle simulator + integration helper
def _bicycle_step(x, y, psi, v, delta, wheelbase, dt):
    """Single Euler step of the kinematic bicycle model."""
    x   += v * np.cos(psi) * dt
    y   += v * np.sin(psi) * dt
    psi += (v / wheelbase) * np.tan(delta) * dt
    psi  = (psi + np.pi) % (2 * np.pi) - np.pi
    return x, y, psi

def _run_simulation(waypoints, speed, wheelbase, Ld, delta_max, dt,
                    x0=0.0, y0=0.2, psi0=0.05, max_steps=800):
    """
    Closed-loop Pure Pursuit simulation.
    Advances the waypoint search window forward to avoid steering back to
    already-passed waypoints — consistent with standard Pure Pursuit usage.
    Returns (xs, ys) trajectory arrays.
    """
    x, y, psi = x0, y0, psi0
    xs, ys = [x], [y]
    start = 0
    for _ in range(max_steps):
        pos = np.array([x, y])
        # Advance start index: never look back at waypoints already within Ld
        while start < len(waypoints) - 1 and               np.linalg.norm(waypoints[start] - pos) < Ld:
            start += 1
        sub_wps = waypoints[start:]
        local_idx = select_lookahead_point(sub_wps, pos, Ld)
        wp = sub_wps[local_idx]
        if (start + local_idx == len(waypoints) - 1 and
                np.linalg.norm(wp - pos) < 1.0):
            break
        alpha = compute_alpha(wp, pos, psi)
        delta = compute_steering(alpha, Ld, wheelbase, delta_max)
        x, y, psi = _bicycle_step(x, y, psi, speed, delta, wheelbase, dt)
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)


In [27]:
# ── Autograder tests — Integration (5 pts) ──────────────────────────────── #
# DO NOT EDIT

from scipy.interpolate import interp1d

Ld = 5.0   # lookahead distance (matches waypoint spacing)

xs, ys = _run_simulation(
    WAYPOINTS, SPEED, WHEELBASE, Ld, DELTA_MAX, DT,
    x0=0.0, y0=0.4, psi0=0.08
)

# The vehicle must reach at least 90 % of the path length
reached_x = xs[-1]
assert reached_x >= 0.9 * WAYPOINTS[-1, 0], (
    f"Vehicle only reached x={reached_x:.1f} m; "
    f"must reach >= {0.9*WAYPOINTS[-1,0]:.1f} m"
)

# Lateral deviation must stay <= 1.5 m throughout the covered range
_path_y = interp1d(WAYPOINTS[:, 0], WAYPOINTS[:, 1],
                   kind="linear", fill_value="extrapolate")
mask = (xs >= WAYPOINTS[0, 0]) & (xs <= WAYPOINTS[-1, 0])
lat_err = np.abs(ys[mask] - _path_y(xs[mask]))
max_err = float(lat_err.max())
assert max_err <= 1.5, (
    f"Max lateral error {max_err:.3f} m exceeds 1.5 m limit"
)

# ── Visualisation (for your own inspection — not graded) ─────────────────
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(WAYPOINTS[:, 0], WAYPOINTS[:, 1], "k--", lw=1.5, label="Reference path")
ax.plot(xs, ys, "b-", lw=2, label="Pure Pursuit trajectory")
ax.scatter(*WAYPOINTS.T, s=25, c="k", zorder=5)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Pure Pursuit")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("pure_pursuit_result.png", dpi=120)
plt.close()

print(f"Integration test passed ✓  (max lateral error = {max_err:.3f} m)")
print("Trajectory plot saved to  pure_pursuit_result.png")


Integration test passed ✓  (max lateral error = 0.410 m)
Trajectory plot saved to  pure_pursuit_result.png


---
# ☆ OPTIONAL — Week 7: Localisation & LiDAR  *(20 pts total)*

> These tasks are **optional** but graded. They cover topics from the Week 7 lecture that did not have a dedicated lab.  
> They are independent of one another — complete any subset in any order.

### Topics covered
| Task | Topic | Lecture slides |
|------|-------|----------------|
| A | Odometry drift accumulation | Slides 14–17 |
| B | Sensor fusion: predict–correct (1-D Kalman) | Slides 18, 24 |
| C | 2-D ICP scan matching (match → minimise → update) | Slides 47–49 |
| D | LiDAR point-cloud de-skewing | Slides 37–38 |



## Optional Task A — Odometry drift accumulation  *(5 pts)*

Wheel odometry integrates small motion estimates.  
Even a tiny constant velocity bias grows without bound — this is **drift**.

Implement dead-reckoning: starting from `pos0`, integrate `velocities` (m/s) with time step `dt`.  
Each velocity reading has additive Gaussian noise with standard deviation `noise_std`.

$$x_{k+1} = x_k + (v_k + \epsilon_k)\cdot dt, \qquad \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

**Signature:** `simulate_odometry(pos0, velocities, dt, noise_std, rng) → np.ndarray`


In [28]:
def simulate_odometry(pos0: float,
                      velocities: np.ndarray,
                      dt: float,
                      noise_std: float,
                      rng: np.random.Generator) -> np.ndarray:
    """
    Simulate 1-D dead-reckoning with additive velocity noise.

    Parameters
    ----------
    pos0      : float           Starting position (metres).
    velocities: np.ndarray (N,) True velocity at each step (m/s).
    dt        : float           Time step (s).
    noise_std : float           Std-dev of velocity noise (m/s).
    rng       : np.random.Generator  Seeded RNG — use rng.normal() for noise.

    Returns
    -------
    np.ndarray, shape (N+1,)
        Estimated position at each time step including t=0.
        positions[0] = pos0.
    """
    velocities = np.asarray(velocities, dtype=float)
    noise = rng.normal(0.0, noise_std, size=velocities.shape)
    steps = (velocities + noise) * dt
    return np.concatenate(([pos0], pos0 + np.cumsum(steps)))


In [29]:
# ── Autograder tests — Task A (5 pts) ───────────────────────────────────── #
# DO NOT EDIT

rng = np.random.default_rng(42)

# Test A1: zero noise, constant velocity → exact integration
vels = np.ones(100) * 5.0   # 5 m/s for 100 steps
pos = simulate_odometry(0.0, vels, dt=0.1, noise_std=0.0, rng=rng)
assert pos.shape == (101,), f"Expected shape (101,), got {pos.shape}"
np.testing.assert_allclose(pos[0], 0.0, atol=1e-9,
                            err_msg="pos[0] must equal pos0")
np.testing.assert_allclose(pos[-1], 50.0, atol=1e-6,
                            err_msg="Zero noise, 5 m/s × 100 × 0.1 s = 50 m")

# Test A2: output is monotonically different from zero-noise when noise is added
rng2 = np.random.default_rng(7)
pos_noisy = simulate_odometry(0.0, vels, dt=0.1, noise_std=0.5, rng=rng2)
assert pos_noisy.shape == (101,), "Shape must be (N+1,)"
assert not np.allclose(pos_noisy, pos),     "Noisy trajectory must differ from noiseless one"

# Test A3: drift — bias of 0.1 m/s over 200 steps of 0.5 s accumulates ~10 m
rng3 = np.random.default_rng(99)
bias_vels = np.zeros(200) + 0.1
pos_bias = simulate_odometry(0.0, bias_vels, dt=0.5, noise_std=0.0, rng=rng3)
np.testing.assert_allclose(pos_bias[-1], 0.1 * 200 * 0.5, atol=1e-6,
                            err_msg="Pure bias should integrate exactly")

# Visualisation
fig, ax = plt.subplots(figsize=(8, 3))
t = np.arange(101) * 0.1
ax.plot(t, pos, "k-", label="No noise")
ax.plot(t, pos_noisy, "r--", alpha=0.7, label="With noise (σ=0.5 m/s)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Estimated position (m)")
ax.set_title("Task A — Odometry Drift")
ax.legend()
plt.tight_layout()
plt.savefig("task_a_odometry.png", dpi=120)
plt.close()

print("Task A — all assertions passed ✓")


Task A — all assertions passed ✓


## Optional Task B — Sensor fusion: 1-D predict–correct  *(5 pts)*

The fundamental loop in sensor fusion is **predict** with motion, then **correct** with an external measurement.

Here we implement a minimal 1-D Kalman filter to fuse odometry (high-rate, drifts) with GPS (low-rate, noisy but absolute).

### Equations

**Predict:**
$$x^- = x + v \cdot dt, \qquad P^- = P + Q$$

**Correct** (when a GPS fix `z` arrives):
$$K = \frac{P^-}{P^- + R}, \qquad x = x^- + K(z - x^-), \qquad P = (1-K)P^-$$

where $Q$ is the process noise variance and $R$ is the GPS measurement noise variance.

**Signatures:**
- `kalman_predict(x, P, v, dt, Q) → (x_pred, P_pred)`
- `kalman_correct(x_pred, P_pred, z, R) → (x_est, P_est)`


In [30]:
def kalman_predict(x: float, P: float,
                   v: float, dt: float,
                   Q: float):
    """
    Kalman prediction step (constant-velocity, 1-D).

    Parameters
    ----------
    x  : float  Current state estimate (position, metres).
    P  : float  Current state variance (m^2).
    v  : float  Velocity from odometry (m/s).
    dt : float  Time step (s).
    Q  : float  Process noise variance (m^2).

    Returns
    -------
    (x_pred, P_pred) : (float, float)
    """
    x_pred = x + v * dt
    P_pred = P + Q
    return x_pred, P_pred


def kalman_correct(x_pred: float, P_pred: float,
                   z: float, R: float):
    """
    Kalman correction step.

    Parameters
    ----------
    x_pred : float  Predicted state.
    P_pred : float  Predicted variance.
    z      : float  GPS measurement (metres).
    R      : float  GPS measurement noise variance (m^2).

    Returns
    -------
    (x_est, P_est) : (float, float)
    """
    K = P_pred / (P_pred + R)
    x_est = x_pred + K * (z - x_pred)
    P_est = (1.0 - K) * P_pred
    return x_est, P_est


In [31]:
# ── Autograder tests — Task B (5 pts) ───────────────────────────────────── #
# DO NOT EDIT

# Test B1: predict — basic integration
x_p, P_p = kalman_predict(0.0, 1.0, v=5.0, dt=0.1, Q=0.01)
np.testing.assert_allclose(x_p, 0.5, atol=1e-9, err_msg="Predict: x_pred = x + v*dt")
np.testing.assert_allclose(P_p, 1.01, atol=1e-9, err_msg="Predict: P_pred = P + Q")

# Test B2: correct — perfect measurement → x must move toward z
x_e, P_e = kalman_correct(0.5, 1.0, z=0.8, R=0.1)
assert 0.5 < x_e < 0.8, "Corrected estimate must move toward measurement"
assert 0 < P_e < 1.0, "Posterior variance must be smaller than prior"

# Test B3: Kalman gain bounded in (0, 1)
K = P_p / (P_p + 0.1)
assert 0 < K < 1, "Kalman gain K must be in (0, 1)"

# Test B4: zero measurement noise → fully trust GPS
x_e2, P_e2 = kalman_correct(10.0, 1.0, z=5.0, R=1e-12)
np.testing.assert_allclose(x_e2, 5.0, atol=1e-4,
                            err_msg="With R≈0, estimate should snap to measurement")

# Test B5: end-to-end — run 200 steps, GPS every 10 steps
np.random.seed(0)
x_true = 0.0
x_est, P_est = 0.0, 4.0
errs = []
for k in range(200):
    # True dynamics
    x_true += 5.0 * DT + np.random.normal(0, 0.05)
    # Predict
    x_est, P_est = kalman_predict(x_est, P_est, v=5.0, dt=DT, Q=0.1)
    # GPS update every 10 steps
    if k % 10 == 0:
        z = x_true + np.random.normal(0, 0.5)
        x_est, P_est = kalman_correct(x_est, P_est, z, R=0.25)
    errs.append(abs(x_est - x_true))
rmse = np.sqrt(np.mean(np.array(errs)**2))
assert rmse < 1.0, f"RMSE {rmse:.3f} m is too large; fusion should outperform odometry alone"

print(f"Task B — all assertions passed ✓  (fusion RMSE = {rmse:.3f} m)")


Task B — all assertions passed ✓  (fusion RMSE = 0.400 m)


## Optional Task C — 2-D ICP scan matching  *(5 pts)*

**Iterative Closest Point (ICP)** aligns two point clouds by iterating:
1. **Match** — for each source point, find the nearest target point.
2. **Minimise** — compute the optimal rotation **R** and translation **t** via SVD.
3. **Update** — apply the transform and repeat.

You implement **one iteration** of the 2-D case.

### SVD-based optimal rotation

Given matched pairs $\{(s_i, t_i)\}$ with centroids $\bar{s}, \bar{t}$:

$$H = \sum_i (s_i - \bar{s})^\top (t_i - \bar{t}), \qquad [U, S, V^\top] = \text{SVD}(H)$$
$$R = V U^\top, \qquad t = \bar{t} - R\bar{s}$$

**Signature:** `icp_step(source, target) → (R, t, mean_dist)`


In [32]:
def icp_step(source: np.ndarray,
             target: np.ndarray):
    """
    One iteration of 2-D Iterative Closest Point.

    Parameters
    ----------
    source : np.ndarray, shape (N, 2)  Points to align.
    target : np.ndarray, shape (M, 2)  Reference point cloud.

    Returns
    -------
    R         : np.ndarray (2, 2)  Optimal rotation matrix.
    t         : np.ndarray (2,)    Optimal translation vector.
    mean_dist : float              Mean distance between matched pairs.

    Algorithm
    ---------
    1. For each source point, find its nearest neighbour in target.
    2. Compute centroids of source and matched-target subsets.
    3. Build the cross-covariance matrix H.
    4. SVD: H = U S Vt. Set R = Vt.T @ U.T.
       Handle reflection: if det(R) < 0, negate the last column of Vt.T.
    5. t = target_centroid - R @ source_centroid.
    6. mean_dist = mean of Euclidean distances between matched pairs.
    """
    source = np.asarray(source, dtype=float)
    target = np.asarray(target, dtype=float)

    # 1. Nearest-neighbour matching
    d2 = np.sum((source[:, None, :] - target[None, :, :]) ** 2, axis=2)
    matched = target[np.argmin(d2, axis=1)]

    # 2. Centroids
    s_bar = source.mean(axis=0)
    t_bar = matched.mean(axis=0)

    # 3. Cross-covariance
    H = (source - s_bar).T @ (matched - t_bar)

    # 4. SVD and reflection handling
    U, S, Vt = np.linalg.svd(H)
    V = Vt.T
    R = V @ U.T
    if np.linalg.det(R) < 0:
        V[:, -1] *= -1
        R = V @ U.T

    # 5. Translation
    t = t_bar - R @ s_bar

    # 6. Mean matched distance
    mean_dist = float(np.mean(np.linalg.norm(source - matched, axis=1)))
    return R, t, mean_dist


In [33]:
# ── Autograder tests — Task C (5 pts) ───────────────────────────────────── #
# DO NOT EDIT

from numpy.testing import assert_allclose

# Well-separated 2-D target cloud (inter-point distances >> any test offset)
target_pts = np.array([[0., 0.], [10., 0.], [10., 8.], [0., 8.], [5., 4.]], dtype=float)

# Test C1: identity — source already aligned → R≈I, t≈0, mean_dist≈0
R_id, t_id, d_id = icp_step(target_pts.copy(), target_pts.copy())
assert_allclose(R_id, np.eye(2), atol=1e-6, err_msg="Aligned clouds: R should be I")
assert_allclose(t_id, np.zeros(2), atol=1e-6, err_msg="Aligned clouds: t should be 0")
assert d_id < 1e-6, "Aligned clouds: mean_dist should be ~0"

# Test C2: pure translation — offset is small relative to inter-point spacing
#          so nearest-neighbour matching is correct in one step
offset = np.array([1.5, -0.8])
source_shifted = target_pts + offset
R_t, t_t, d_t = icp_step(source_shifted, target_pts)
reconstructed = (R_t @ source_shifted.T).T + t_t
assert_allclose(reconstructed, target_pts, atol=1e-5,
                err_msg="Pure translation: reconstructed points must match target")

# Test C3: small rotation (10°) — correct NN matching, recoverable in one step
theta = np.deg2rad(10)
Rot = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
source_rot = (Rot @ target_pts.T).T
R_r, t_r, d_r = icp_step(source_rot, target_pts)
reconstructed_r = (R_r @ source_rot.T).T + t_r
assert_allclose(reconstructed_r, target_pts, atol=1e-4,
                err_msg="10° rotation: reconstructed must match target after 1 ICP step")
assert_allclose(R_r @ Rot, np.eye(2), atol=1e-4,
                err_msg="R_r should be the inverse (transpose) of the applied rotation")

# Test C4: output types and shapes
assert R_id.shape == (2, 2), "R must be (2, 2)"
assert t_id.shape == (2,), "t must be (2,)"
assert isinstance(d_id, float) or np.ndim(d_id) == 0, "mean_dist must be scalar"

print("Task C — all assertions passed ✓")


Task C — all assertions passed ✓


## Optional Task D — LiDAR point-cloud de-skewing  *(5 pts)*

A spinning 3-D LiDAR acquires points over ~100 ms. During that time the shuttle moves,
so points captured at different times come from different sensor poses.  
**De-skewing** corrects each point to a common reference pose.

For a 1-D corridor model, each point `p_i` is captured at time `t_i`.  
The sensor position at time `t` is `x_sensor(t) = x0 + v * t` (constant velocity).  
The **reference pose** is `x_sensor(t_ref) = x0 + v * t_ref`.

To move point `p_i` from its acquisition pose to the reference pose, subtract the ego-motion:

$$p^{\text{corrected}}_i = p_i - v \cdot (t_i - t_{\text{ref}})$$

(This is a 1-D simplification of the full 6-DoF transform used in practice.)

**Signature:** `deskew(points, timestamps, t_ref, velocity) → np.ndarray`


In [34]:
def deskew(points: np.ndarray,
           timestamps: np.ndarray,
           t_ref: float,
           velocity: float) -> np.ndarray:
    """
    De-skew a 1-D LiDAR scan for constant vehicle velocity.

    Context
    -------
    A naive LiDAR point-cloud reconstruction assigns the *end-of-scan* pose to
    every point.  This creates apparent 'shearing': the wall appears at
    different world positions depending on when each beam was fired.
    De-skewing subtracts the positional drift between each point's true
    acquisition time and the chosen reference time.

    Parameters
    ----------
    points     : np.ndarray (N,)
        Apparent world-frame point positions computed assuming all beams were
        fired at ``t_ref`` (i.e., the sheared / uncorrected cloud).
    timestamps : np.ndarray (N,)
        Actual acquisition time of each point (seconds).
    t_ref      : float
        Reference time to which all points are corrected (seconds).
        Typically the start or end of the scan.
    velocity   : float
        Sensor velocity (m/s, constant during the scan, positive = forward).

    Returns
    -------
    np.ndarray (N,)
        De-skewed point positions: each point moved to the reference pose.

    Formula
    -------
    corrected[i] = points[i] - velocity * (t_ref - timestamps[i])
    """
    points = np.asarray(points, dtype=float)
    timestamps = np.asarray(timestamps, dtype=float)
    return points - velocity * (t_ref - timestamps)


In [35]:
# ── Autograder tests — Task D (5 pts) ───────────────────────────────────── #
# DO NOT EDIT

# Simulate a 64-beam LiDAR sweep at 10 Hz (100 ms total)
n_pts     = 64
t_scan    = np.linspace(0.0, 0.1, n_pts)   # acquisition timestamps (s)
v_shuttle = 14.0                             # m/s (~50 km/h)
t_ref_val = t_scan[-1]                       # reference = end-of-scan

# A flat wall sits at world_x = 50 m.
# A naive reconstruction (treating all beams as fired at t_ref) yields:
#   apparent_world_x[i] = sensor_x(t_ref) + range[i]
#                       = v*t_ref + (50 - v*t[i])   (sensor moves forward)
#                       = 50 + v*(t_ref - t[i])
# → earlier beams appear further away: the wall looks sheared/sloped.
true_wall_x = 50.0
raw_sheared = true_wall_x + v_shuttle * (t_ref_val - t_scan)   # sheared cloud

# After de-skewing, all corrected positions must equal true_wall_x
corrected = deskew(raw_sheared, t_scan, t_ref=t_ref_val, velocity=v_shuttle)

np.testing.assert_allclose(corrected, true_wall_x, atol=1e-6,
    err_msg="De-skewed points must all map to the true wall position")

# Test D2: zero velocity → no correction needed
raw_static = np.ones(20) * 30.0
corrected_static = deskew(raw_static, np.linspace(0, 0.1, 20), 0.05, 0.0)
np.testing.assert_allclose(corrected_static, raw_static, atol=1e-9,
    err_msg="Zero velocity: deskewing should not change the points")

# Test D3: shape preserved
assert corrected.shape == raw_sheared.shape, "Output shape must match input shape"

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.scatter(t_scan, raw_sheared, s=10, c="r", label="Raw (sheared)")
ax1.axhline(true_wall_x, c="k", ls="--", label="True wall position")
ax1.set_title("Before de-skewing"); ax1.set_xlabel("Time (s)"); ax1.set_ylabel("Apparent world x (m)")
ax1.legend(fontsize=8)

ax2.scatter(t_scan, corrected, s=10, c="b", label="De-skewed")
ax2.axhline(true_wall_x, c="k", ls="--", label="True wall position")
ax2.set_title("After de-skewing"); ax2.set_xlabel("Time (s)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("task_d_deskew.png", dpi=120)
plt.close()

print("Task D — all assertions passed ✓")
print(f"  Spread before de-skewing: {np.ptp(raw_sheared):.3f} m")
print(f"  Spread after  de-skewing: {np.ptp(corrected):.6f} m  (should be ~0)")


Task D — all assertions passed ✓
  Spread before de-skewing: 1.400 m
  Spread after  de-skewing: 0.000000 m  (should be ~0)
